# Training a Convolutional Neural Network on MNIST

## What We Will Build

In this notebook, we train a **Convolutional Neural Network (CNN)** to recognize handwritten digits from the **MNIST** dataset. By the end, you will understand:

- What a CNN is and why it works well for image recognition
- How to load and preprocess image data for a neural network
- How to define, compile, and train a CNN using TensorFlow/Keras
- How to evaluate the trained model with metrics like accuracy, confusion matrices, and classification reports
- How to save a trained model for later use

### What is MNIST?

MNIST is a dataset of 70,000 handwritten digit images (0 through 9), each 28x28 pixels in grayscale. It is the classic benchmark for image classification and a perfect starting point for learning about CNNs.

### What is a CNN?

A **Convolutional Neural Network** is a type of neural network designed specifically for processing grid-like data such as images. Unlike a standard neural network that treats each pixel independently, a CNN uses small sliding filters (called *kernels*) that scan across the image to detect patterns like edges, curves, and shapes. This makes CNNs far more effective at understanding spatial structure in images.

### Our Architecture (High-Level)

We will build a deliberately simple CNN with:
- **2 convolutional layers** (8 filters each) to detect patterns
- **2 pooling layers** to reduce spatial dimensions
- **2 dense (fully connected) layers** to classify based on detected features
- **Dropout layers** for regularization
- A **softmax output** that gives a probability for each of the 10 digits

This architecture is intentionally small so that the feature maps and activations are easy to visualize and understand.

## Section 2: Data Loading

Our MNIST data comes from Michael Nielsen's "Neural Networks and Deep Learning" book. It is stored as a compressed pickle file (`mnist.pkl.gz`). This format packages the data as Python objects serialized with Python 2, so we need to use `encoding='latin1'` when loading.

The file contains three datasets:
- **Training set** (50,000 images) -- used to train the model
- **Validation set** (10,000 images) -- used to monitor training progress
- **Test set** (10,000 images) -- used for final evaluation

Each image is stored as a flat vector of 784 values (28 x 28 pixels), with pixel intensities already normalized to the range [0, 1].

In [ ]:
import gzip
import pickle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the MNIST data from the Nielsen pickle file
with gzip.open('/Users/saurabh/Downloads/mnist.pkl.gz', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

# Nielsen format: 3-tuple of (images, labels) pairs
# Auto-detect structure in case the file uses a 2-tuple variant
if len(data) == 3:
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = data
    print("Loaded 3-tuple format (train/val/test)")
elif len(data) == 2:
    (x_train, y_train), (x_test, y_test) = data
    # Create validation split from training data
    x_val, y_val = x_train[50000:], y_train[50000:]
    x_train, y_train = x_train[:50000], y_train[:50000]
    print("Loaded 2-tuple format (train/test), split validation from training")

# Print shapes to verify the data loaded correctly
print(f"Training set:   {x_train.shape} images, {y_train.shape} labels")
print(f"Validation set: {x_val.shape} images, {y_val.shape} labels")
print(f"Test set:       {x_test.shape} images, {y_test.shape} labels")
print(f"Label range:    {y_train.min()} to {y_train.max()}")
print(f"Pixel dtype:    {x_train.dtype}, range: [{x_train.min():.3f}, {x_train.max():.3f}]")

## Section 3: Preprocessing

### Why Reshape?

The data is currently stored as flat vectors of 784 values. But a CNN needs to understand the **spatial structure** of the image -- it needs to know which pixels are next to each other. So we reshape each vector back into a 28x28 grid.

We also add a **channel dimension**, making each image shape `(28, 28, 1)`. The "1" means grayscale (one color channel). Color images would have 3 channels (red, green, blue).

### Normalization Check

The Nielsen data is already normalized to [0, 1]. Many MNIST tutorials divide pixel values by 255, but that assumes raw integer pixel values (0-255). **We must NOT divide by 255 again** -- doing so would squash our data to near-zero and the model would fail to learn.

In [ ]:
# CRITICAL: Verify that data is already in [0, 1] range
# Nielsen data is pre-normalized -- do NOT divide by 255
assert x_train.max() <= 1.0, f"Data max is {x_train.max()}, expected <= 1.0"
assert x_train.min() >= 0.0, f"Data min is {x_train.min()}, expected >= 0.0"
print(f"Data range confirmed: [{x_train.min():.3f}, {x_train.max():.3f}] -- no normalization needed")

In [ ]:
# Reshape from flat vectors (N, 784) to 2D images (N, 28, 28, 1)
# The extra dimension of 1 is the grayscale channel
x_train = x_train.reshape(-1, 28, 28, 1)
x_val = x_val.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print(f"Reshaped training set:   {x_train.shape}")
print(f"Reshaped validation set: {x_val.shape}")
print(f"Reshaped test set:       {x_test.shape}")

In [ ]:
# Display 10 sample images to verify the reshape preserved spatial structure
# If digits look correct (not rotated, scrambled, or blank), the reshape is right
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.suptitle("Sample Training Images (verify orientation)")
plt.tight_layout()
plt.show()

### A Note on Labels

Our labels are integers (0 through 9). Some tutorials convert these to **one-hot encoded** vectors, where the label 3 becomes `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. We skip that step because we use `sparse_categorical_crossentropy` as our loss function, which works directly with integer labels. This keeps the code simpler.

## Section 4: Model Architecture

Now we define the CNN. Here is what each layer type does:

### Conv2D (Convolutional Layer)
A convolution layer slides small filters (kernels) across the input image. Each filter learns to detect a specific pattern -- early filters detect simple features like edges and corners, while deeper filters detect more complex shapes. We use 8 filters per layer, each 3x3 pixels.

### MaxPooling2D (Pooling Layer)
After convolution, pooling reduces the spatial dimensions by taking the maximum value in each small region (2x2 in our case). This makes the representation smaller and more robust to small shifts in the input.

### Flatten
Converts the 2D feature maps into a 1D vector so it can be fed into dense layers.

### Dense (Fully Connected Layer)
Every neuron connects to every neuron in the previous layer. These layers learn to classify based on the features extracted by the convolutional layers.

### Dropout
During training, dropout randomly "turns off" a fraction of neurons (25% in our case). This prevents the model from memorizing the training data (overfitting) and forces it to learn more robust features.

### Softmax (Output)
The final layer has 10 neurons (one per digit). Softmax converts the raw outputs into probabilities that sum to 1. The digit with the highest probability is the model's prediction.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Build the CNN using Sequential API (layers stacked one after another)
# Every layer has an explicit name= parameter for Phase 2 activation extraction
model = keras.Sequential([
    # Input shape: 28x28 grayscale images
    layers.Input(shape=(28, 28, 1)),

    # First convolutional block: 8 filters detect basic patterns (edges, curves)
    layers.Conv2D(8, kernel_size=(3, 3), activation='relu', name='conv2d_1'),
    layers.MaxPooling2D(pool_size=(2, 2), name='maxpool_1'),

    # Second convolutional block: 8 filters detect higher-level features
    layers.Conv2D(8, kernel_size=(3, 3), activation='relu', name='conv2d_2'),
    layers.MaxPooling2D(pool_size=(2, 2), name='maxpool_2'),

    # Flatten the 2D feature maps into a 1D vector
    layers.Flatten(name='flatten'),

    # Dense layers for classification
    layers.Dense(128, activation='relu', name='dense_1'),
    layers.Dropout(0.25, name='dropout_1'),     # Regularization: drop 25% of neurons
    layers.Dense(64, activation='relu', name='dense_2'),
    layers.Dropout(0.25, name='dropout_2'),

    # Output layer: 10 classes (digits 0-9), softmax gives probabilities
    layers.Dense(10, activation='softmax', name='output'),
])

In [ ]:
# Display the model architecture and parameter counts
model.summary()

### Reading the Summary

The summary shows each layer, its output shape, and how many trainable parameters it has:

- **Conv2D layers**: Parameters = (kernel_height x kernel_width x input_channels x num_filters) + num_filters (bias). For conv2d_1: (3 x 3 x 1 x 8) + 8 = 80 parameters.
- **MaxPooling layers**: No trainable parameters -- they just take the max value.
- **Dense layers**: Parameters = (input_size x output_size) + output_size (bias). The first dense layer has the most parameters because it connects the flattened feature maps to 128 neurons.
- **Dropout layers**: No parameters -- they just randomly zero out neurons during training.
- **Total parameters**: This is a deliberately small model for pedagogical clarity. Production models may have millions of parameters.

## Section 5: Training

### Compilation

Before training, we **compile** the model by specifying:

- **Optimizer: Adam** -- An adaptive learning rate optimizer that adjusts how much to update each weight. It is the most popular default choice because it works well without manual tuning.
- **Loss: sparse_categorical_crossentropy** -- Measures how wrong the model's predictions are. "Sparse" means our labels are integers (not one-hot). "Categorical crossentropy" is the standard loss for multi-class classification.
- **Metrics: accuracy** -- The fraction of correctly classified images. We track this alongside the loss.

### Training Process

Training works in **epochs** (passes through the entire dataset). In each epoch:
1. The training data is split into **batches** (128 images at a time)
2. For each batch, the model makes predictions, calculates the loss, and adjusts its weights (backpropagation)
3. After all batches, the model is evaluated on the **validation set** to check for overfitting

We train for 15 epochs, which is enough for this model to converge on MNIST.

In [ ]:
# Compile the model: define how it learns
model.compile(
    optimizer='adam',                          # Adaptive learning rate
    loss='sparse_categorical_crossentropy',    # For integer labels
    metrics=['accuracy']                       # Track classification accuracy
)

In [ ]:
# Train the model
# - epochs=15: pass through the full training set 15 times
# - batch_size=128: process 128 images at a time
# - validation_data: evaluate on validation set after each epoch
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_data=(x_val, y_val),
    verbose=1
)

In [ ]:
# Plot training curves to visualize learning progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves (lower is better)
ax1.plot(history.history['loss'], label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves (higher is better)
ax2.plot(history.history['accuracy'], label='Training Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Evaluation

Training accuracy tells us how well the model learned the training data, but the real test is how it performs on **data it has never seen**. That is what the held-out test set is for.

We evaluate using:
1. **Overall test accuracy** -- a single number summarizing performance
2. **Confusion matrix** -- shows which digits get confused with each other
3. **Classification report** -- per-digit precision, recall, and F1 score

In [ ]:
# Evaluate on the test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

In [ ]:
# Get predictions for the full test set
y_pred = model.predict(x_test).argmax(axis=1)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Build and display the confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Digit')
plt.ylabel('True Digit')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

### How to Read the Confusion Matrix

- **Diagonal cells** (top-left to bottom-right) show **correct** predictions. Larger numbers on the diagonal are better.
- **Off-diagonal cells** show **errors** -- the row is the true label and the column is what the model predicted. For example, if the cell at row 4, column 9 has a high value, the model is frequently misclassifying 4s as 9s.
- Common confusions include 4/9 (similar top curves), 3/5 (similar shapes), and 7/1 (both have a vertical stroke).

In [ ]:
# Classification report: per-digit precision, recall, and F1-score
# - Precision: of all images predicted as digit X, what fraction actually are X?
# - Recall: of all actual digit X images, what fraction did the model find?
# - F1-score: harmonic mean of precision and recall
print(classification_report(y_test, y_pred, digits=3))

### Interpreting the Results

Look at the per-digit F1 scores. Digits that are visually similar (like 4 and 9, or 3 and 5) tend to have slightly lower scores. The overall accuracy should be above 95%, which is impressive given our deliberately simple model with only 8 filters per convolutional layer.

## Section 7: Model Export

We save the trained model as an `.h5` file (HDF5 format). This is a single portable file that contains:
- The model architecture (layer definitions)
- The trained weights (what the model learned)
- The optimizer state (for resuming training if needed)

The backend API (Phase 2) will load this exact file to serve predictions and to extract intermediate layer activations for visualization. This is why we gave every layer an explicit `name=` parameter -- the backend references layers by name.

In [ ]:
import os

# Create the model directory and save
os.makedirs('model', exist_ok=True)
model.save('model/mnist_cnn.h5')
print("Model saved to model/mnist_cnn.h5")
print(f"File size: {os.path.getsize('model/mnist_cnn.h5') / 1024:.1f} KB")

In [ ]:
# Verify: reload the model from disk and confirm it works
loaded = keras.models.load_model('model/mnist_cnn.h5')
loaded.summary()

In [ ]:
# Verify Phase 2 compatibility: build a multi-output model for activation extraction
# This is the exact pattern the backend will use to visualize what happens inside the CNN

layer_outputs = [layer.output for layer in loaded.layers
                 if not isinstance(layer, keras.layers.InputLayer)]
activation_model = keras.Model(inputs=loaded.input, outputs=layer_outputs)

# Test with one sample image
sample = x_test[0:1]
activations = activation_model.predict(sample)

print(f"Activation extraction works: {len(activations)} layer outputs")
for i, act in enumerate(activations):
    # Skip InputLayer (index 0) when mapping to layer names
    layer_name = loaded.layers[i + 1].name if (i + 1) < len(loaded.layers) else "unknown"
    print(f"  Layer {i} ({layer_name}): shape {act.shape}")

### Next Steps

The trained model (`model/mnist_cnn.h5`) is now ready. In Phase 2, the backend API will:
1. Load this model at startup
2. Accept digit images from users and return predictions
3. Extract activations from each layer to visualize what the CNN "sees" at every step

This is what makes the project educational -- users will not just see a prediction, they will see *how* the CNN arrived at that prediction by examining the feature maps and activations layer by layer.